# Parallel Heston Calibration with Parfun + QuantLib

The **Heston model** is a widely-used stochastic volatility model in quantitative finance.
Calibrating it requires non-linear optimization against a grid of market-observed option prices.

A common technique is **multi-start optimization**: run the calibration from several different
initial guesses and pick the best result. Each run is independent, making this
embarrassingly parallel.

Parfun parallelises this with minimal code changes.

## Program Structure

![Program Structure](Simple_Heston_Calibration_Parfun_flowchart.svg)

# Native Python Implementation

In [1]:
import time

import numpy as np
import QuantLib as ql

sequential_runtime = None


def calibrate_heston_once(initial_guess):
    """Calibrate a Heston model with a given initial guess.
    Uses a dense strike x maturity grid to create a realistic workload.
    Returns calibrated parameters and error."""
    spot = 100
    risk_free = 0.01
    dividend = 0.00

    rf_handle = ql.YieldTermStructureHandle(ql.FlatForward(0, ql.NullCalendar(), risk_free, ql.Actual365Fixed()))
    div_handle = ql.YieldTermStructureHandle(ql.FlatForward(0, ql.NullCalendar(), dividend, ql.Actual365Fixed()))

    process = ql.HestonProcess(
        rf_handle, div_handle,
        ql.QuoteHandle(ql.SimpleQuote(spot)),
        initial_guess[0], initial_guess[1], initial_guess[2],
        initial_guess[3], initial_guess[4],
    )

    model = ql.HestonModel(process)
    engine = ql.AnalyticHestonEngine(model)

    strikes = [70, 80, 85, 90, 95, 100, 105, 110, 115, 120, 130]
    maturities = [ql.Period(m, ql.Months) for m in [3, 6, 12, 18, 24]]
    base_vols = {
        70: 0.30, 80: 0.26, 85: 0.24, 90: 0.22, 95: 0.21,
        100: 0.20, 105: 0.21, 110: 0.22, 115: 0.24, 120: 0.26, 130: 0.30,
    }

    helpers = []
    for mat in maturities:
        for k in strikes:
            h = ql.HestonModelHelper(
                mat, ql.NullCalendar(), spot, k,
                ql.QuoteHandle(ql.SimpleQuote(base_vols[k])),
                rf_handle, div_handle,
            )
            h.setPricingEngine(engine)
            helpers.append(h)

    model.calibrate(helpers, ql.LevenbergMarquardt(), ql.EndCriteria(5000, 500, 1e-8, 1e-8, 1e-8))
    error = sum(h.calibrationError() for h in helpers)

    return list(model.params()), error


def calibrate_heston(initial_guesses):
    """Calibrate Heston model for each initial guess."""
    return [calibrate_heston_once(g) for g in initial_guesses]


def main():
    global sequential_runtime

    initial_guesses = [
        [0.1, 1.0, 0.05, 0.3, -0.5],
        [0.2, 0.5, 0.04, 0.4, -0.3],
        [0.05, 2.0, 0.07, 0.2, -0.7],
        [0.15, 1.5, 0.06, 0.25, -0.4],
        [0.08, 0.8, 0.03, 0.35, -0.6],
        [0.12, 1.2, 0.08, 0.15, -0.2],
        [0.18, 0.3, 0.05, 0.5, -0.8],
        [0.06, 1.8, 0.04, 0.28, -0.45],
    ]

    start = time.time()
    results = calibrate_heston(initial_guesses)
    sequential_runtime = time.time() - start

    print(f"Sequential: {sequential_runtime:.2f}s")


if __name__ == "__main__":
    main()

IOStream.flush timed out
IOStream.flush timed out


Sequential: 148.90s


# Parfun Implementation

In [2]:
import time

import numpy as np
import QuantLib as ql

import parfun as pf

parallel_runtime = None


def calibrate_heston_once(initial_guess):
    """Calibrate a Heston model with a given initial guess.
    Uses a dense strike x maturity grid to create a realistic workload.
    Returns calibrated parameters and error."""
    spot = 100
    risk_free = 0.01
    dividend = 0.00

    rf_handle = ql.YieldTermStructureHandle(ql.FlatForward(0, ql.NullCalendar(), risk_free, ql.Actual365Fixed()))
    div_handle = ql.YieldTermStructureHandle(ql.FlatForward(0, ql.NullCalendar(), dividend, ql.Actual365Fixed()))

    process = ql.HestonProcess(
        rf_handle, div_handle,
        ql.QuoteHandle(ql.SimpleQuote(spot)),
        initial_guess[0], initial_guess[1], initial_guess[2],
        initial_guess[3], initial_guess[4],
    )

    model = ql.HestonModel(process)
    engine = ql.AnalyticHestonEngine(model)

    strikes = [70, 80, 85, 90, 95, 100, 105, 110, 115, 120, 130]
    maturities = [ql.Period(m, ql.Months) for m in [3, 6, 12, 18, 24]]
    base_vols = {
        70: 0.30, 80: 0.26, 85: 0.24, 90: 0.22, 95: 0.21,
        100: 0.20, 105: 0.21, 110: 0.22, 115: 0.24, 120: 0.26, 130: 0.30,
    }

    helpers = []
    for mat in maturities:
        for k in strikes:
            h = ql.HestonModelHelper(
                mat, ql.NullCalendar(), spot, k,
                ql.QuoteHandle(ql.SimpleQuote(base_vols[k])),
                rf_handle, div_handle,
            )
            h.setPricingEngine(engine)
            helpers.append(h)

    model.calibrate(helpers, ql.LevenbergMarquardt(), ql.EndCriteria(5000, 500, 1e-8, 1e-8, 1e-8))
    error = sum(h.calibrationError() for h in helpers)

    return list(model.params()), error


@pf.parfun(
    split=pf.per_argument(initial_guesses=pf.py_list.by_chunk),
    combine_with=pf.py_list.concat,
    fixed_partition_size=1,
)
def calibrate_heston(initial_guesses):
    """Calibrate Heston model for each initial guess."""
    return [calibrate_heston_once(g) for g in initial_guesses]


def main():
    global parallel_runtime

    initial_guesses = [
        [0.1, 1.0, 0.05, 0.3, -0.5],
        [0.2, 0.5, 0.04, 0.4, -0.3],
        [0.05, 2.0, 0.07, 0.2, -0.7],
        [0.15, 1.5, 0.06, 0.25, -0.4],
        [0.08, 0.8, 0.03, 0.35, -0.6],
        [0.12, 1.2, 0.08, 0.15, -0.2],
        [0.18, 0.3, 0.05, 0.5, -0.8],
        [0.06, 1.8, 0.04, 0.28, -0.45],
    ]

    with pf.set_parallel_backend_context("scaler_local", n_workers=4):
        start = time.time()
        results = calibrate_heston(initial_guesses)
        parallel_runtime = time.time() - start

    print(f"Parallel:   {parallel_runtime:.2f}s")


if __name__ == "__main__":
    main()

[INFO]2026-03-31 10:58:29+0800: logging to ('/dev/stdout',)
[INFO]2026-03-31 10:58:29+0800: ObjectStorageServer: start and listen to tcp://127.0.0.1:50237
[INFO]2026-03-31 10:58:29+0800: ObjectStorageServer: started
[INFO]2026-03-31 10:58:30+0800: logging to ('/dev/stdout',)
[INFO]2026-03-31 10:58:30+0800: use event loop: builtin
[INFO]2026-03-31 10:58:30+0800: ConfigController: scheduler_address = tcp://127.0.0.1:51415
[INFO]2026-03-31 10:58:30+0800: ConfigController: object_storage_address = tcp://127.0.0.1:50237
[INFO]2026-03-31 10:58:30+0800: ConfigController: monitor_address = None
[INFO]2026-03-31 10:58:30+0800: ConfigController: protected = True
[INFO]2026-03-31 10:58:30+0800: ConfigController: max_number_of_tasks_waiting = -1
[INFO]2026-03-31 10:58:30+0800: ConfigController: client_timeout_seconds = 60
[INFO]2026-03-31 10:58:30+0800: ConfigController: worker_timeout_seconds = 60
[INFO]2026-03-31 10:58:30+0800: ConfigController: object_retention_seconds = 60
[INFO]2026-03-31 10:

# Diff: Native vs Parfun

In [3]:
import re
import difflib
from IPython.display import display, HTML

native_code = In[1]
parfun_code = In[2]

differ = difflib.HtmlDiff(wrapcolumn=90)
table = differ.make_table(
    native_code.splitlines(),
    parfun_code.splitlines(),
    fromdesc="Native Python",
    todesc="With Parfun",
    context=False,
)

# Strip unwanted columns: navigation links, line numbers, and nowrap attribute
table = re.sub(r'<td class="diff_next"[^>]*>.*?</td>', '', table)
table = re.sub(r'<th class="diff_next"[^>]*>.*?</th>', '', table)
table = re.sub(r'<td class="diff_header"[^>]*>.*?</td>', '', table)
table = table.replace('colspan="2"', '')
table = table.replace(' nowrap="nowrap"', '')
table = re.sub(r'(\s*<colgroup></colgroup>){6}',
               '\n        <colgroup></colgroup> <colgroup></colgroup>', table)

css = """<style>
    table.diff { font-family: monospace; font-size: 10px; border-collapse: collapse; width: 100%; }
    table.diff td { padding: 2px 8px; white-space: pre-wrap; word-break: break-all; text-align: left !important; }
    table.diff th { padding: 6px 8px; text-align: center; background-color: #f0f0f0; font-size: 14px; }
    td.diff_add { background-color: #e6ffec; }
    td.diff_chg { background-color: #fff8c5; }
    td.diff_sub { background-color: #ffebe9; }
    span.diff_add { background-color: #ccffd8; }
    span.diff_chg { background-color: #fff2a8; }
    span.diff_sub { background-color: #ffc0c0; }
</style>"""

display(HTML(css + table))

Native Python,With Parfun
import time,import time
,
import numpy as np,import numpy as np
import QuantLib as ql,import QuantLib as ql
,
sequential_runtime = None,import parfun as pf
,
,parallel_runtime = None
,
,


# Speedup

In [4]:
print(f"Sequential: {sequential_runtime:.2f}s")
print(f"Parallel:   {parallel_runtime:.2f}s")
print(f"Speedup:    {sequential_runtime / parallel_runtime:.2f}x")

Sequential: 148.90s
Parallel:   70.42s
Speedup:    2.11x
